# COLMAP

Using **pycolmap** library to perform SfM (Structure from Motion) on images retrieved from video frames.

## Sparse Reconstruction

Using pycolmap to run the SfM pipeline and generate the camera position and points

In [1]:
import pathlib
import pycolmap

output_path= pathlib.Path("../results/colmap_lego")
image_dir= pathlib.Path("../data/images/lego/")

output_path.mkdir()
database_path = output_path / "database.db"

pycolmap.extract_features(database_path, image_dir)
pycolmap.match_exhaustive(database_path)
maps = pycolmap.incremental_mapping(database_path, image_dir, output_path)
maps[0].write(output_path)

## Visualizing 

Visualizing the SfM point clouds, removing line frustums

In [3]:
import sys
sys.path.append("../utils")
from colmap import load_colmap_model, create_open3d_geometries_from_colmap, save_colmap_geometries_ply
import open3d as o3d
import numpy as np

MODEL_DIR = "../results/colmap_lego/0"   # change to "1" for the second model
PLY_OUT   = "../results/colmap_lego/reconstruction.ply"

# Load model and base point cloud
cams, images, points = load_colmap_model(MODEL_DIR)
geometries = create_open3d_geometries_from_colmap(MODEL_DIR, scale_frustum=0.05)
pcd = geometries[0] if geometries else o3d.geometry.PointCloud()

# Create camera center spheres instead of line frustums
spheres = []
sphere_radius = 0.01
for img_id in sorted(images.keys()):
    info = images[img_id]
    qvec = info["qvec"]  # qw, qx, qy, qz
    tvec = info["tvec"]
    qw, qx, qy, qz = qvec
    q = np.array([qw, qx, qy, qz], dtype=float)
    w, x, y, z = q
    R = np.array([
        [1 - 2 * (y * y + z * z),     2 * (x * y - z * w),     2 * (x * z + y * w)],
        [    2 * (x * y + z * w), 1 - 2 * (x * x + z * z),     2 * (y * z - x * w)],
        [    2 * (x * z - y * w),     2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ], dtype=float)
    C = (-R.T @ tvec).ravel()
    sph = o3d.geometry.TriangleMesh.create_sphere(radius=sphere_radius)
    sph.translate(C)
    sph.compute_vertex_normals()
    sph.paint_uniform_color([0.1, 0.6, 0.9])
    spheres.append(sph)

print(f"Point cloud: {len(pcd.points)} points; Camera spheres: {len(spheres)}")

# Save points + camera spheres
out = save_colmap_geometries_ply([pcd] + spheres, PLY_OUT)
print(f"Saved PLY to: {out}")

# Visualize
o3d.visualization.draw_geometries(
    [pcd] + spheres,
    window_name="COLMAP Points and Camera Centers",
    width=1280,
    height=720,
)

Point cloud: 102 points; Camera spheres: 2
Saved PLY to: ..\results\colmap_lego\reconstruction.ply
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 


## Converting to mesh

Converting the points to texturized mesh

In [3]:
import sys
import numpy as np
import open3d as o3d
sys.path.append("../utils")
from colmap import load_colmap_model
from mesh_transforming import poisson_reconstruction_from_point_cloud

# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_DIR   = "../results/colmap_apple/0"
MESH_OUTPUT = "../results/colmap_apple/mesh.ply"
# ──────────────────────────────────────────────────────────────────────────────

# Load only the reconstructed 3-D points (no camera markers or frustums)
_, _, points = load_colmap_model(MODEL_DIR)
if not points:
    raise RuntimeError("No 3-D points in model — run sparse reconstruction first.")

pts  = np.vstack([v["xyz"] for v in points.values()])
cols = np.vstack([v["rgb"] for v in points.values()])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
pcd.colors = o3d.utility.Vector3dVector(cols.astype(np.float64) / 255.0)
print(f"Loaded {len(pcd.points):,} reconstructed points")

pcd = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=1.0)[0]

# ── Mesh parameters (edit these) ─────────────────────────────────────────────

# Normal estimation
NORMAL_RADIUS    = 0.05   # search radius; decrease for denser, tighter clouds
NORMAL_MAX_NN    = 30     # max neighbours used per point
NORMAL_ORIENT_K  = 100    # neighbourhood size for consistent orientation

# Poisson solver
POISSON_DEPTH    = 10      # octree depth: 8=coarser/faster, 10=finer/slower
POISSON_SCALE    = 1.1    # bounding-box padding factor
LINEAR_FIT       = True  # True = smoother but slower iso-surface interpolation

# ──────────────────────────────────────────────────────────────────────────────

mesh = poisson_reconstruction_from_point_cloud(
    pcd,
    output_mesh_file      = MESH_OUTPUT,
    depth                 = POISSON_DEPTH,
    width                 = 0,
    scale                 = POISSON_SCALE,
    linear_fit            = LINEAR_FIT
)

o3d.visualization.draw_geometries(
    [mesh],
    width=1280,
    height=720,
)

Loaded 3,283 reconstructed points
Estimating normals...
Running Poisson surface reconstruction...
[Open3D DEBUG] Input Points / Samples: 2903 / 2841
[Open3D DEBUG] #   Got kernel density: 0.01399993896484375 (s), 194.26171875 (MB) / 249.09765625 (MB) / 292 (MB)
[Open3D DEBUG] #     Got normal field: 0.010999917984008789 (s), 198.26953125 (MB) / 249.09765625 (MB) / 292 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 3.192289e-04 / 9.267214e-01
[Open3D DEBUG] #       Finalized tree: 0.026999950408935547 (s), 206.8125 (MB) / 249.09765625 (MB) / 292 (MB)
[Open3D DEBUG] #  Set FEM constraints: 0.03400015830993652 (s), 203.0546875 (MB) / 249.09765625 (MB) / 292 (MB)
[Open3D DEBUG] #Set point constraints: 0.004999876022338867 (s), 203.1328125 (MB) / 249.09765625 (MB) / 292 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 229006 / 117472 / 144249
[Open3D DEBUG] Memory Usage: 203.133 MB
[Open3D DEBUG] # Linear system solved: 0.10699987411499023 (s), 208.65234375 (MB) / 249.097656